# DSA4156 Lab: Inmon vs. Kimball on Sample Superstore
## Template notebook

| | |
|---|---|
| **Group name** | *fill in* |
| **Members** | *fill in* |

You will build two data warehouses from the same source data, in one PostgreSQL database:

- `staging`: raw and cleaned copies of the source files
- `inmon`: a **3NF** enterprise warehouse
- `kimball`: a **denormalized** dimensional model (star or snowflake schema)

**Rules**
- Cells marked `# TODO` are yours to complete. You may add as many cells as you need.
- You may transform with pandas, SQL, or both, but every step must run from this notebook.
- Before submitting, run **Kernel > Restart & Run All** and save the notebook *with outputs*.
- Your data model diagrams go in the PDF report, not here.

## 0. Setup

**Requirements**
- PostgreSQL **15 or newer** (16 recommended). Version 15 is needed for `UNIQUE NULLS NOT DISTINCT`.
- Python packages: `pandas`, `sqlalchemy>=2.0`, `psycopg2-binary`

Create an empty database first, for example in `psql`:
```sql
CREATE DATABASE superstore_dw;
```
Then set your connection string below, or set the `DW_URL` environment variable.
**Do not commit real passwords** in the notebook you submit.

In [ ]:
import os, re
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy import types as sa_types

DATA_DIR = Path("data")
DW_URL = os.getenv("DW_URL", "postgresql+psycopg2://postgres:<password>@localhost:5432/superstore_dw")

engine = create_engine(DW_URL, future=True)

def run_sql(sql: str) -> None:
    """Execute one or more SQL statements (DDL or DML) in a single transaction."""
    with engine.begin() as conn:
        conn.exec_driver_sql(sql)

def q(sql: str, **params) -> pd.DataFrame:
    """Run a SELECT and return the result as a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

def table_counts(schema: str) -> pd.DataFrame:
    """Row count of every table (partitioned parents included, partitions excluded) in a schema."""
    tables = q("""
        SELECT c.relname AS table_name
        FROM pg_class c JOIN pg_namespace n ON n.oid = c.relnamespace
        WHERE n.nspname = :s AND c.relkind IN ('r', 'p') AND NOT c.relispartition
        ORDER BY 1""", s=schema)["table_name"]
    rows = [(t, int(q(f'SELECT count(*) AS n FROM "{schema}"."{t}"')["n"][0])) for t in tables]
    return pd.DataFrame(rows, columns=["table_name", "row_count"])

print(q("SELECT version()")["version"][0])

PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35228, 64-bit


## 1. Extract

The source system gave us three files (originally three sheets of one workbook):

| File | Contents |
|---|---|
| `orders.csv` | One row per order line (flat, denormalized, 21 columns) |
| `returns.csv` | Orders that were returned |
| `people.csv` | The regional manager of each region |

We read everything **as text** so that nothing is silently converted, then copy the raw data into a
`staging` schema. Staging keeps an untouched copy of the source so the warehouse can always be rebuilt.

In [10]:
raw_orders  = pd.read_csv(DATA_DIR / "orders.csv",  dtype=str, keep_default_na=False)
raw_returns = pd.read_csv(DATA_DIR / "returns.csv", dtype=str, keep_default_na=False)
raw_people  = pd.read_csv(DATA_DIR / "people.csv",  dtype=str, keep_default_na=False)

def snake(col: str) -> str:
    return re.sub(r"[^0-9a-z]+", "_", col.strip().lower()).strip("_")

run_sql("CREATE SCHEMA IF NOT EXISTS staging;")
for name, df in [("orders_raw", raw_orders), ("returns_raw", raw_returns), ("people_raw", raw_people)]:
    df.rename(columns=snake).to_sql(name, engine, schema="staging", if_exists="replace", index=False)

print({n: len(d) for n, d in [("orders", raw_orders), ("returns", raw_returns), ("people", raw_people)]})
raw_orders.head()

{'orders': 9994, 'returns': 800, 'people': 4}


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0,41.9136
1,2,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0,219.582
2,3,CA-2017-138688,6/12/2017,6/16/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0,6.8714
3,4,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
4,5,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164


In [17]:
raw_orders['Order ID'].duplicated().sum()

np.int64(4985)

### 1.1 Profile the source: find the functional dependencies

Before designing any model, test which columns determine which. `A -> B` holds when every value of `A`
maps to exactly one value of `B`. The count below is the number of `A` values that break the rule.
These results are the evidence for your normalization decisions in the report.

In [ ]:
orders = raw_orders.rename(columns=snake)

def fd_violations(df: pd.DataFrame, lhs, rhs: str) -> int:
    """Number of distinct LHS values that map to more than one RHS value."""
    counts = df.groupby(lhs, dropna=False)[rhs].nunique(dropna=False)
    return int((counts > 1).sum())

In [ ]:
from itertools import permutations
#lhs->rhs

def check_all_fds(df: pd.DataFrame) -> pd.DataFrame:
    columns = list(df.columns)
    results = []

    for lhs, rhs in permutations(columns, 2):
        violations = fd_violations(df, lhs, rhs)
        results.append(
            {
                "Determinant": lhs,
                "Dependent": rhs,
                "Violations": violations,
                "Holds": violations == 0,
            }
        )

    return pd.DataFrame(results, columns=["Determinant", "Dependent", "Violations", "Holds"])

In [38]:
# Example
#print("order_id -> customer_id violations:", fd_violations(orders, "order_id", "customer_id"))

# TODO: test every dependency you think might hold, e.g. customer -> segment, product -> category,
#       postal code -> city, city -> state, state -> region, and so on.
#       Build a small table of results (determinant, dependent, violations) to put in your report.

#testing all columns for orders
pd.set_option("display.max_rows", None)
order_dependency_df = check_all_fds(orders)

In [36]:
order_dependency_df[order_dependency_df["Violations"] > 0].sort_values("Violations", ascending=False)

,Determinant,Dependent,Violations,Holds
20,order_id,row_id,2471,False
39,order_id,profit,2471,False
36,order_id,sales,2471,False
35,order_id,product_name,2469,False
32,order_id,product_id,2469,False
34,order_id,sub_category,2346,False
37,order_id,quantity,2237,False
340,sales,row_id,1986,False
341,sales,order_id,1985,False
343,sales,ship_date,1985,False


In [41]:
#reset display max rows
pd.reset_option("display.max_rows")

In [43]:
#check for uniqueness of ids
def check_id_uniqueness(df: pd.DataFrame) -> pd.DataFrame:
    """Check uniqueness of every column whose name contains 'ID'.

    Columns are matched if their name contains 'ID' (case-insensitive). A
    column is considered unique when it has no duplicate non-null values.
    'n duplicates' counts the number of extra occurrences beyond the first,
    i.e. total rows minus the number of distinct non-null values.

    Returns a DataFrame with columns: "column name", "unique", "n duplicates".
    """
    id_cols = [col for col in df.columns if "id" in str(col).lower()]

    results = []
    for col in id_cols:
        non_null = df[col].dropna()
        n_duplicates = int(non_null.duplicated().sum())
        results.append(
            {
                "column name": col,
                "unique": n_duplicates == 0,
                "n duplicates": n_duplicates,
            }
        )

    return pd.DataFrame(
        results, columns=["column name", "unique", "n duplicates"]
    )

In [44]:
check_id_uniqueness(orders)

,column name,unique,n duplicates
0,row_id,True,0
1,order_id,False,4985
2,customer_id,False,9201
3,product_id,False,8132


The criteria for the helper function used is that the column must simply have 'id' in the name, however some columns CAN serve as identifiers without explicitly having 'id' in the name such as 'Region' for a normalized database with a Region table (Region is also present in the 'People' table). For such cases, it is manually checked. 

In [ ]:
# TODO: look for data quality problems before you design anything. For example:
#   - Is (order_id, product_id) unique? What should the order-line key be?
#   - Do all postal codes have 5 digits? Are any missing?
#   - Do all IDs follow the same format?
#   - Does returns.csv contain duplicates?
# The source contains several such problems. Your report must list the ones you found and how you handled them.


### **Checking one-to-one mapping between order_id and order_date**

In some case for the orders table/dataframe, for a row with a value of OrderID, A, its Order Date is B such that if A occurs in one row as the ID, its Date is B. It may repeat but its always A-B for any row A or B occurs in. We can verify this below:

In [47]:
n_ids = orders["order_id"].nunique()
n_dates = orders["order_date"].nunique()
n_pairs = len(orders[["order_id", "order_date"]].drop_duplicates())

print(n_ids == n_dates == n_pairs)

False


In [61]:
print(order_dependency_df[(order_dependency_df["Determinant"] == "order_id")&(order_dependency_df["Dependent"] == "order_date")])
print(order_dependency_df[(order_dependency_df["Determinant"] == "order_date")& (order_dependency_df["Dependent"] == "order_id")])

   Determinant   Dependent  Violations  Holds
21    order_id  order_date           0   True
   Determinant Dependent  Violations  Holds
41  order_date  order_id        1014  False


This turns out to be false, or not true both ways. Every order_id maps to exactly one order_date but some dates have more than one order_id hence a many-to-one relationship between order_id and order_date. Altough trivial, its still worth checking for data integrity. 

### **Checking Missing Values**

In [62]:
def report_missing_vals(df: pd.DataFrame) -> pd.DataFrame:
    missing = df.isna().sum()
    report = pd.DataFrame(
        {
            "column": missing.index,
            "missing values": missing.values,
            "percentage": (missing.values / len(df)) * 100,
        }
    )
    return report

In [63]:
report_missing_vals(orders)

,column,missing values,percentage
0,row_id,0,0.0
1,order_id,0,0.0
2,order_date,0,0.0
3,ship_date,0,0.0
4,ship_mode,0,0.0
5,customer_id,0,0.0
6,customer_name,0,0.0
7,segment,0,0.0
8,country,0,0.0
9,city,0,0.0


### **Checking Postal Codes**
By definition, postal codes must be 5-digits in the USA. 

In [72]:
len(orders[orders["postal_code"].astype(str).str.len() != 5])

449

**Profiling findings** (summarize here, and in more detail in your report)

*TODO: write your findings and the design decision each one leads to.*

## 2. Transform: shared cleaning

Clean the data **once**, then build both warehouses from the same clean staging tables.
That way, any difference between the two warehouses comes from the model, not from the cleaning.

Load your result to `staging.orders_clean`, `staging.returns_clean`, and `staging.people_clean` with proper types.

In [ ]:
clean = orders.copy()

# TODO: convert types (dates, numbers), trim text, fix the problems you found in profiling.

returns_clean = raw_returns.rename(columns=snake)   # TODO: clean
people_clean  = raw_people.rename(columns=snake)    # TODO: clean

# TODO: add assert statements that stop the pipeline if the data is still invalid, e.g.
# assert clean["row_id"].is_unique

clean.to_sql("orders_clean", engine, schema="staging", if_exists="replace", index=False,
             dtype={"order_date": sa_types.Date, "ship_date": sa_types.Date})   # TODO: add the other types
returns_clean.to_sql("returns_clean", engine, schema="staging", if_exists="replace", index=False)
people_clean.to_sql("people_clean", engine, schema="staging", if_exists="replace", index=False)

table_counts("staging")

## 3. Inmon architecture: 3NF enterprise warehouse

### 3.1 Physical model (DDL)

Write the `CREATE TABLE` statements for your 3NF model. This DDL implements the physical data model in your report.
Use **at least three PostgreSQL-specific features** and explain each one in the report
(for example identity columns, `CHECK` constraints, `UNIQUE NULLS NOT DISTINCT`, indexes, schemas, comments).

One example table is given so you can see the style. Replace or extend it.

In [ ]:
INMON_DDL = r"""
DROP SCHEMA IF EXISTS inmon CASCADE;
CREATE SCHEMA inmon;

CREATE TABLE inmon.segment (
    segment_id   SMALLINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    segment_name VARCHAR(20) NOT NULL UNIQUE
);

-- TODO: the rest of your 3NF tables, constraints, and indexes
"""
run_sql(INMON_DDL)

### 3.2 Load the 3NF core

Load parent tables before child tables so that foreign keys resolve.

In [ ]:
INMON_LOAD = r"""
INSERT INTO inmon.segment (segment_name)
SELECT DISTINCT segment FROM staging.orders_clean;

-- TODO: load every other table
"""
run_sql(INMON_LOAD)
table_counts("inmon")

In [ ]:
# Reconciliation: prove the warehouse holds exactly what the source holds.
# TODO: change the table and column names to match your model, and add more checks.
checks = {
    # "order lines = source rows": q("SELECT count(*) n FROM inmon.<your_line_table>")["n"][0] == len(clean),
    # "total sales matches": np.isclose(float(q("SELECT sum(sales) s FROM inmon.<your_line_table>")["s"][0]), clean["sales"].sum()),
}
assert all(checks.values()), checks
checks

## 4. Kimball architecture: dimensional model

### 4.1 Declare the model

*TODO: fill in before writing any SQL.*

- **Business process:**
- **Grain (one fact row = ?):**
- **Dimensions:**
- **Facts (measures):**

In [ ]:
KIMBALL_DDL = r"""
DROP SCHEMA IF EXISTS kimball CASCADE;
CREATE SCHEMA kimball;

-- TODO: dimension tables, then the fact table, constraints, and indexes
"""
run_sql(KIMBALL_DDL)

### 4.2 Load dimensions, then the fact table

Tip: PostgreSQL's `generate_series()` can build a complete date dimension in a single statement.

In [ ]:
KIMBALL_LOAD = r"""
-- TODO: load dimensions first, then the fact table
"""
run_sql(KIMBALL_LOAD)
table_counts("kimball")

In [ ]:
# TODO: reconciliation checks for the Kimball warehouse (row count, total sales, total profit)


## 5. Same business questions, both warehouses

Answer each question **twice**: once against the Inmon 3NF warehouse (`inmon` schema) and once against
the Kimball model. Both answers must be identical, and they must match the source data.

1. Total sales and profit by region for each year.
2. Return rate (percentage of order lines that belong to a returned order) and returned sales, by product category.
3. Top 5 customers by sales in 2018, with their segment and number of orders.

In [ ]:
Q1_INMON = """
-- TODO
"""
Q1_KIMBALL = """
-- TODO
"""
q1_i, q1_k = q(Q1_INMON), q(Q1_KIMBALL)
# TODO: assert that q1_i and q1_k give the same numbers, and check one of them against pandas on `clean`
q1_k

In [ ]:
Q2_INMON = """
-- TODO
"""
Q2_KIMBALL = """
-- TODO
"""
q2_i, q2_k = q(Q2_INMON), q(Q2_KIMBALL)
# TODO: assert that q2_i and q2_k give the same numbers, and check one of them against pandas on `clean`
q2_k

In [ ]:
Q3_INMON = """
-- TODO
"""
Q3_KIMBALL = """
-- TODO
"""
q3_i, q3_k = q(Q3_INMON), q(Q3_KIMBALL)
# TODO: assert that q3_i and q3_k give the same numbers, and check one of them against pandas on `clean`
q3_k

In [ ]:
def join_count(sql): return len(re.findall(r"\bJOIN\b", sql, flags=re.I))

comparison = pd.DataFrame({
    "question": ["Q1", "Q2", "Q3"],
    "joins_inmon_3NF": [join_count(s) for s in (Q1_INMON, Q2_INMON, Q3_INMON)],
    "joins_kimball": [join_count(s) for s in (Q1_KIMBALL, Q2_KIMBALL, Q3_KIMBALL)],
})
comparison

## 6. Evidence for the report

Export the tables you will show in your report. Screenshots of query results are also fine.
For the ER diagrams of the *resulting* warehouses, you can generate them from the live database with
pgAdmin (ERD Tool), DBeaver, or DataGrip.

In [ ]:
out = Path("report_outputs"); out.mkdir(exist_ok=True)
table_counts("inmon").to_csv(out / "inmon_row_counts.csv", index=False)
table_counts("kimball").to_csv(out / "kimball_row_counts.csv", index=False)
comparison.to_csv(out / "join_comparison.csv", index=False)
# TODO: export anything else you need (sample rows, query results, profiling table)
sorted(p.name for p in out.iterdir())